# Lab 10: Red neuronal con Keras

Usamos Keras para construir y entrenar redes neuronales de forma más cómoda que desde cero.
Exploramos las funciones de activación principales y comparamos arquitecturas sobre un dataset de clasificación.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import make_classification, make_moons, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from tensorflow import keras
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

## 1. Funciones de activación disponibles en Keras

Graficamos las funciones de activación más comunes para entender visualmente su comportamiento.
Esto es útil para decidir cuál usar en cada capa según el problema.

In [ ]:
x = np.linspace(-4, 4, 300)

activaciones = {
    'ReLU':    keras.activations.relu(x).numpy(),
    'Sigmoid': keras.activations.sigmoid(x).numpy(),
    'Tanh':    keras.activations.tanh(x).numpy(),
    'ELU':     keras.activations.elu(x, alpha=1.0).numpy(),
    'SELU':    keras.activations.selu(x).numpy(),
    'Softplus':keras.activations.softplus(x).numpy(),
}

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()

for i, (nombre, y_act) in enumerate(activaciones.items()):
    axes[i].plot(x, y_act, linewidth=2)
    axes[i].axhline(0, color='gray', linewidth=0.7, linestyle='--')
    axes[i].axvline(0, color='gray', linewidth=0.7, linestyle='--')
    axes[i].set_title(nombre)
    axes[i].set_xlabel('z')
    axes[i].grid(alpha=0.2)

plt.suptitle('Funciones de activación en Keras')
plt.tight_layout()
plt.show()

### Resumen de características

| Función | Rango | Media cero | Gradiente | Uso típico |
|---------|-------|------------|-----------|------------|
| Lineal  | $(-\infty,+\infty)$ | sí | constante | capa de salida (regresión) |
| Sigmoid | $(0, 1)$ | no | se aplana | salida binaria |
| Tanh    | $(-1, 1)$ | sí | se aplana | capas ocultas (clásico) |
| ReLU    | $[0,+\infty)$ | no | grande | capas ocultas (estándar hoy) |
| ELU     | $(-\alpha,+\infty)$ | ~sí | suave | capas ocultas (alternativa) |
| SELU    | auto-normaliza | sí | suave | redes profundas densas |

## 2. Dataset de clasificación binaria (make_classification)

Preparamos un dataset de clasificación binaria con 20 features y lo normalizamos antes de entrenarlo.

In [ ]:
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_redundant=4, random_state=7
)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=7)

# Siempre es buena práctica escalar antes de una red neuronal
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

print(f'Train: {X_tr.shape}, Test: {X_te.shape}')

## 3. Tres formas de construir un modelo en Keras

El enunciado mostraba varias formas. Las tres son equivalentes; la más limpia para este lab es la tercera.

In [ ]:
# Forma 1: capa de Activación explícita
modelA = keras.models.Sequential([
    keras.layers.Dense(32, input_shape=(20,)),
    keras.layers.Activation('relu'),
    keras.layers.Dense(16),
    keras.layers.Activation('tanh'),
    keras.layers.Dense(1, activation='sigmoid'),
])

# Forma 2: argumento activation en Dense
modelB = keras.models.Sequential([
    keras.layers.Dense(32, activation='relu', input_shape=(20,)),
    keras.layers.Dense(16, activation='tanh'),
    keras.layers.Dense(1, activation='sigmoid'),
])

print('ModelA:')
modelA.summary()
print('\nModelB (equivalente):')
modelB.summary()

## 4. Comparación de funciones de activación en capas ocultas

Entrenamos el mismo modelo con distintas funciones de activación en las capas ocultas y comparamos el accuracy y la curva de pérdida.

In [ ]:
def construir_modelo(activacion_oculta, n_features):
    modelo = keras.models.Sequential([
        keras.layers.Dense(64, activation=activacion_oculta, input_shape=(n_features,)),
        keras.layers.Dense(32, activation=activacion_oculta),
        keras.layers.Dense(1, activation='sigmoid'),
    ])
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo


activaciones_comparar = ['relu', 'tanh', 'sigmoid', 'elu']
historiales = {}
resultados = []

for act in activaciones_comparar:
    tf.random.set_seed(42)
    m = construir_modelo(act, X_tr.shape[1])
    hist = m.fit(
        X_tr, y_tr,
        epochs=50, batch_size=64,
        validation_split=0.1,
        verbose=0
    )
    historiales[act] = hist
    acc_te = m.evaluate(X_te, y_te, verbose=0)[1]
    resultados.append({'Activación': act, 'Acc Test': round(acc_te, 4)})
    print(f'{act:10s}: test accuracy = {acc_te:.4f}')

display(pd.DataFrame(resultados))

In [ ]:
# Curvas de pérdida en validación para comparar convergencia
plt.figure(figsize=(9, 5))
for act, hist in historiales.items():
    plt.plot(hist.history['val_loss'], label=act)
plt.xlabel('Época')
plt.ylabel('Val Loss (cross-entropy)')
plt.title('Convergencia según función de activación')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print('ReLU y ELU suelen converger más rápido que Tanh y Sigmoid en capas ocultas.')

## 5. Modelo final con la mejor activación

Tomamos ReLU (que suele funcionar mejor en general) y entrenamos más épocas para afinar el resultado.

In [ ]:
tf.random.set_seed(42)
modelo_final = keras.models.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(X_tr.shape[1],)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])

modelo_final.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

hist_final = modelo_final.fit(
    X_tr, y_tr,
    epochs=100, batch_size=64,
    validation_split=0.1,
    verbose=0
)

acc_tr_final = modelo_final.evaluate(X_tr, y_tr, verbose=0)[1]
acc_te_final = modelo_final.evaluate(X_te, y_te, verbose=0)[1]
print(f'Train accuracy: {acc_tr_final:.4f}')
print(f'Test  accuracy: {acc_te_final:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_final.history['loss'], label='Train')
ax1.plot(hist_final.history['val_loss'], label='Val')
ax1.set_title('Loss')
ax1.set_xlabel('Época')
ax1.legend()
ax1.grid(alpha=0.2)

ax2.plot(hist_final.history['accuracy'], label='Train')
ax2.plot(hist_final.history['val_accuracy'], label='Val')
ax2.set_title('Accuracy')
ax2.set_xlabel('Época')
ax2.legend()
ax2.grid(alpha=0.2)

plt.suptitle('Modelo final (ReLU, 64-32-16-1)')
plt.tight_layout()
plt.show()

## 6. Clasificación con make_moons

`make_moons` genera datos no linealmente separables: dos lunas entrelazadas.
Con solo 2 features es fácil visualizar las fronteras que aprende la red.

In [ ]:
X_moons, y_moons = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_moons, y_moons, test_size=0.2, stratify=y_moons, random_state=42
)

scaler_m = StandardScaler()
X_tr_m = scaler_m.fit_transform(X_tr_m)
X_te_m  = scaler_m.transform(X_te_m)

# Visualizacion de los datos
plt.figure(figsize=(6, 4))
for cl, color in zip([0, 1], ['steelblue', 'darkorange']):
    mask = y_tr_m == cl
    plt.scatter(X_tr_m[mask, 0], X_tr_m[mask, 1], s=15,
                color=color, label=f'Clase {cl}', alpha=0.7)
plt.legend()
plt.title('make_moons - datos de entrenamiento (escalados)')
plt.grid(alpha=0.2)
plt.show()

# Modelo Keras para make_moons
tf.random.set_seed(42)
modelo_moons = keras.models.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(2,)),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])
modelo_moons.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

hist_moons = modelo_moons.fit(
    X_tr_m, y_tr_m,
    epochs=100, batch_size=32,
    validation_split=0.1,
    verbose=0
)

acc_moons = modelo_moons.evaluate(X_te_m, y_te_m, verbose=0)[1]
print(f'Test accuracy make_moons: {acc_moons:.4f}')

# Curva de entrenamiento
plt.figure(figsize=(8, 3))
plt.plot(hist_moons.history['loss'], label='Train loss')
plt.plot(hist_moons.history['val_loss'], label='Val loss')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.title('Convergencia - make_moons')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

print('La red aprende la frontera curva entre las dos lunas con solo dos capas ocultas.')

## 7. Clasificación multiclase con load_digits

Para clasificación multiclase (10 dígitos del 0 al 9), la capa de salida usa `softmax`
y la función de pérdida cambia a `sparse_categorical_crossentropy`.
Las imágenes son de 8x8 píxeles aplanadas a 64 features.

In [ ]:
digits = load_digits()
X_dig, y_dig = digits.data, digits.target

print(f'Dataset: {X_dig.shape}, clases: {np.unique(y_dig)}')

# Mostramos algunos dígitos
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for d in range(10):
    idx = np.where(y_dig == d)[0][0]
    axes[0, d].imshow(digits.images[idx], cmap='gray_r')
    axes[0, d].set_title(str(d))
    axes[0, d].axis('off')
    idx2 = np.where(y_dig == d)[0][3]
    axes[1, d].imshow(digits.images[idx2], cmap='gray_r')
    axes[1, d].axis('off')
plt.suptitle('Ejemplos del dataset Digits')
plt.tight_layout()
plt.show()

X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_dig, y_dig, test_size=0.2, stratify=y_dig, random_state=42
)

scaler_d = StandardScaler()
X_tr_d = scaler_d.fit_transform(X_tr_d)
X_te_d  = scaler_d.transform(X_te_d)

# Modelo multiclase: softmax + sparse_categorical_crossentropy
tf.random.set_seed(42)
modelo_digits = keras.models.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(64,)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax'),
])
modelo_digits.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
modelo_digits.summary()

hist_digits = modelo_digits.fit(
    X_tr_d, y_tr_d,
    epochs=100, batch_size=32,
    validation_split=0.1,
    verbose=0
)

acc_tr_d = modelo_digits.evaluate(X_tr_d, y_tr_d, verbose=0)[1]
acc_te_d = modelo_digits.evaluate(X_te_d, y_te_d, verbose=0)[1]
print(f'Train accuracy digits: {acc_tr_d:.4f}')
print(f'Test  accuracy digits: {acc_te_d:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(hist_digits.history['loss'], label='Train')
ax1.plot(hist_digits.history['val_loss'], label='Val')
ax1.set_title('Loss - load_digits')
ax1.set_xlabel('Época')
ax1.legend()
ax1.grid(alpha=0.2)

ax2.plot(hist_digits.history['accuracy'], label='Train')
ax2.plot(hist_digits.history['val_accuracy'], label='Val')
ax2.set_title('Accuracy - load_digits')
ax2.set_xlabel('Época')
ax2.legend()
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print('Con softmax la salida es una distribucion de probabilidad sobre las 10 clases.')

## Conclusiones

- Keras abstrae toda la implementación de backpropagation, pero es importante entender qué hay debajo (Lab 9).
- ReLU domina en capas ocultas porque evita el problema de gradientes que se aplanan en los extremos (vanishing gradient).
- Sigmoid solo se usa hoy en la capa de salida de clasificación binaria; softmax para multiclase.
- `make_moons` demuestra que la red puede aprender fronteras no lineales con pocas capas.
- Para clasificación multiclase (`load_digits`, 10 clases) solo cambia la capa de salida: `softmax` + `sparse_categorical_crossentropy`.
- Siempre conviene escalar los datos antes de pasar por una red neuronal.